# Sample Exam: Python Associate Exam - VoltBike Innovations

VoltBike Innovations is a leading company in the electric bicycle (e-bike) industry, specializing in the design and manufacture of high-performance e-bikes. The company is dedicated to advancing urban mobility solutions by delivering state-of-the-art e-bikes with features such as varying motor powers, advanced battery capacities, and efficient charge systems.

Recently, VoltBike Innovations has encountered some challenges in managing production costs while ensuring high levels of customer satisfaction. These issues have led to increased production expenses and variability in costs, impacting overall profitability.

You are part of the data analysis team tasked with providing actionable insights to help VoltBike Innovations address these challenges.

# Task 1

Before you can start any analysis, you need to confirm that the data is accurate and reflects what you expect to see. 

It is known that there are some issues with the `production_data` table, and the data team have provided the following data description. 

Write a query to return data matching this description. You must match all column names and description criteria.
</br>
Create a cleaned version of the dataframe.

- You should start with the data in the file `ebike_data.csv`.
- Your output should be a dataframe named clean_data.
- All column names and values should match the table below.
</br>

| Column Name         | Criteria                                                                                         |
|----------------------|--------------------------------------------------------------------------------------------------|
| bike_type            | Categorical. Type of e-bike. ['standard', 'folding', 'mountain', 'road']. <br> Missing values should be replaced with 'standard'. |
| frame_material       | Categorical. Material of the e-bike frame. ['aluminum', 'steel', 'carbon fiber']. <br> Missing values should be replaced with 'unknown'. |
| production_cost      | Continuous. Cost of production (in USD). <br> Missing values should be replaced with median. |
| assembly_time        | Continuous. Time taken for assembly (in minutes). <br> Missing values should be replaced with mean. |
| top_speed            | Continuous. Maximum speed of the e-bike (in km/h). <br> Missing values should be replaced with mean. |
| battery_type         | Categorical. Type of battery used. ['li-ion', 'nimh', 'lead acid']. <br> Missing values should be replaced with 'other'. |
| motor_power          | Continuous. Power output of the motor (in watts). <br> Missing values should be replaced with median. |
| customer_score       | Continuous. Customer satisfaction score (rating on a scale of 1 to 10). <br> Missing values should be replaced with mean. |



In [3]:
import pandas as pd
import numpy as np

#import data
orig_data = pd.read_csv('ebike_data.csv')
display(orig_data.sample(10))
display(orig_data.info())
display(orig_data.isna().sum().sum())

#investigate values in each column
#display(orig_data.customer_score.value_counts(dropna=False).sort_index())
    #GOOD:          bike_type, production_cost, assembly_time
    #NEED TO FIX:   frame_material, top_speed, battery_type, motor_power, customer_score???

display(orig_data.customer_score.value_counts(dropna=False).sort_index())


,bike_type,frame_material,production_cost,assembly_time,top_speed,battery_type,motor_power,customer_score
493,road,steel,292.15,59.34,24.82,li-ion,232W,6.76
1979,folding,steel,671.54,95.14,38.91,lead acid,250W,7.59
1918,folding,aluminum,486.04,69.66,21.82,lead acid,301W,5.91
695,standard,steel,690.15,70.80,30.78,li-ion,255W,9.03
1880,road,carbon fiber,273.84,54.07,26.81,lead acid,224W,7.46
1847,road,carbon fiber,696.15,45.62,29.05,nimh,212W,9.62
1110,folding,steel,450.03,73.46,21.39,lead acid,284W,5.47
1891,road,carbon fiber,580.59,71.22,23.53,lead acid,332W,4.21
1116,road,aluminum,515.46,53.20,20.87,nimh,243W,5.81
663,mountain,steel,767.88,97.10,19.95,nimh,313W,9.12


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bike_type        2000 non-null   object 
 1   frame_material   2000 non-null   object 
 2   production_cost  2000 non-null   float64
 3   assembly_time    2000 non-null   float64
 4   top_speed        1940 non-null   float64
 5   battery_type     2000 non-null   object 
 6   motor_power      2000 non-null   object 
 7   customer_score   2000 non-null   float64
dtypes: float64(4), object(4)
memory usage: 125.1+ KB


None

60

customer_score
1.36     1
1.71     1
1.89     1
1.91     1
1.96     1
        ..
10.83    1
10.86    1
10.92    1
10.93    1
11.55    1
Name: count, Length: 645, dtype: int64

In [4]:
#display(orig_data[~orig_data.frame_material.isin(['steel','aluminum','carbon fiber'])])

#make a copy of original dataset to avoid modifying original
data = orig_data.copy()

#Fix values in columns as necessary
    #'frame_material' -- fix some values that are incorrectly formatted; "STEel" --> "steel"
data = data.replace({'frame_material':{'STEel':'steel'}})
    #'top_speed' -- replace null values with mean (~25.0 kph), round to two values to be consistent with other values
data = data.fillna({'top_speed':np.round(data.top_speed.mean(), 2)})
    #'battery_type' -- fix some wrong values; "-" --> "other"
data = data.replace({'battery_type':{'-':'other'}})
    #'motor_power' -- remove "W", change dtype from object to integer
data['motor_power'] = data.motor_power.str.strip('W').astype('int')
    #'customer_score' -- states that values should be on a scale from 1-10 but some values exceed 10;
        #round them down to 10? or replace with mean 'customer_score'? --> ROUNDED THEM DOWN TO 10
for i in data.index:
    if data.loc[i,'customer_score'] > 10:
        data.loc[i,'customer_score'] = 10.0

#display(data.customer_score.value_counts(dropna=False).sort_index())
#display(data.info())


In [5]:
#finalize cleaned data & check
clean_data = data.copy()
display(clean_data.info())
display(clean_data.isna().sum().sum())
display(clean_data.top_speed.value_counts(dropna=False).sort_index().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bike_type        2000 non-null   object 
 1   frame_material   2000 non-null   object 
 2   production_cost  2000 non-null   float64
 3   assembly_time    2000 non-null   float64
 4   top_speed        2000 non-null   float64
 5   battery_type     2000 non-null   object 
 6   motor_power      2000 non-null   int32  
 7   customer_score   2000 non-null   float64
dtypes: float64(4), int32(1), object(3)
memory usage: 117.3+ KB


None

0

2000

# Task 2

You want to understand how different types of e-bikes influence production costs, assembly times, and customer satisfaction.

Calculate the average production_cost, assembly_time, and customer_score grouped by bike_type.

- You should start with the data in the file `ebike_data.csv`.
- Your output should be a data frame named `bike_type_data`.
- It should include the four columns:`bike_type`, `avg_production_cost`, `avg_assembly_time`, and `avg_customer_score`.
- Your answers should be rounded to 2 decimal places.

In [6]:
#start with data in file--"orig_data"--as opposed to using cleaned data from Task 1
#display(orig_data)
#get avgs per 'bike_type'
task2_data = orig_data.groupby('bike_type')[['production_cost','assembly_time','customer_score']].mean().reset_index().rename(
    columns={'production_cost':'avg_production_cost', 'assembly_time':'avg_assembly_time', 
             'customer_score':'avg_customer_score'})
#display(task2_data)
#round avgs to 2 decimals
task2_data = np.round(task2_data, 2)
#finalize & define the desired data
bike_type_data = task2_data.copy()
display(bike_type_data)


,bike_type,avg_production_cost,avg_assembly_time,avg_customer_score
0,folding,499.72,61.40,6.46
1,mountain,507.02,59.79,6.52
2,road,503.02,61.19,6.56
3,standard,489.85,59.81,6.50


# Task 3

In order to proceed with further analysis, you need to understand how key production and satisfaction factors relate to each other. Start by calculating the mean and standard deviation for the following columns: `production_cost` and `customer_score`. These statistics will help in understanding the central tendency and variability of the data related to e-bike production and customer feedback.

Next, calculate the Pearson correlation coefficient between `production_cost` and `customer_score`. This correlation coefficient will provide insights into the strength and direction of the relationship between production costs and customer satisfaction.

- You should start with the data in the file `ebike_data.csv`.
- Calculate the mean and standard deviation for the columns `production_cost` and `customer_score` as: `production_cost_mean`, `production_cost_sd`, `customer_score_mean`, and `customer_score_sd`.
- Calculate the Pearson correlation coefficient between `production_cost` and `customer_score` as `corr_coef`.
- Your output should be a data frame named bike_analysis.
- It should include the columns: `production_cost_mean`, `production_cost_sd`, `customer_score_mean`, `customer_score_sd`, and `corr_coef`.
- Ensure that your answers are rounded to 2 decimal places.


In [7]:
#Use original dataset--"orig_data"
#Find avg, sd of 'production_cost' & 'customer_score'; use "orig_data"
task3_dataI = orig_data[['production_cost','customer_score']].agg(['mean','std'])
#round values to 2 decimals
task3_dataI = np.round(task3_dataI, 2)
#display(task3_dataI)

#Find correlation (Pearson) between 'production_cost', 'customer_score'
task3_dataII = orig_data[['production_cost','customer_score']].corr(method='pearson')
#round values to 2 decimals
task3_dataII = np.round(task3_dataII, 2)
#display(task3_dataII)

#Put desired values into a dataframe & constuct it as specified
task3_dict = {'production_cost_mean':task3_dataI.iloc[0,0], 'production_cost_sd':task3_dataI.iloc[1,0], 
             'customer_score_mean':task3_dataI.iloc[0,1], 'customer_score_sd':task3_dataI.iloc[1,1],
             'corr_coef':task3_dataII.iloc[0,1]}
bike_analysis = pd.DataFrame.from_dict(task3_dict, orient='index').T
display(bike_analysis)


,production_cost_mean,production_cost_sd,customer_score_mean,customer_score_sd,corr_coef
0,500.0,173.34,6.51,1.63,0.48
